In [ ]:
"""
HDB Resale Price Modeling — Extending the Part C Baseline (Revised)
=====================================================================
Part C baseline (this version): floor_area_sqm + town -> LinearRegression

NOTE on this revision: town has far more categories (26) than flat_type (7),
so one-hot encoding it produces a wider, less sparse-friendly feature matrix.
RandomForestRegressor on scipy sparse input with unbounded tree depth is
*very* slow on a single-core machine -- so this version deliberately:
  (a) sets OneHotEncoder(sparse_output=False) to get a dense numpy array, and
  (b) bounds tree depth (max_depth=15, min_samples_leaf=5) for every
      tree-based model -- which also doubles as the overfitting mitigation
      step, since unbounded RF/GB trees on 230K rows memorize noise easily.

This script runs:
  Option 1: + remaining_lease (cleaned to numeric years)
  Option 2: RandomForestRegressor(n_estimators=300) vs 100 -- does more trees help?
  Option 3: Find the single worst-predicted flat in the test set
  Part D:   Stacking (RF + GradientBoosting -> Ridge) vs each model alone
"""

import re
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ----------------------------------------------------------------------------
# 0. Load + clean
# ----------------------------------------------------------------------------
df = pd.read_csv("data/Resale_flat_prices_based_on_registration_date_from_Jan-2017_onwards.csv")

def parse_lease_to_years(lease_str: str) -> float:
    """'61 years 04 months' -> 61.33  (numeric years, for use as a model feature)"""
    years = re.search(r"(\d+)\s*year", lease_str)
    months = re.search(r"(\d+)\s*month", lease_str)
    y = int(years.group(1)) if years else 0
    m = int(months.group(1)) if months else 0
    return y + m / 12

df["remaining_lease_years"] = df["remaining_lease"].apply(parse_lease_to_years)

FEATURES_BASE = ["floor_area_sqm", "town"]
FEATURES_3 = ["floor_area_sqm", "town", "remaining_lease_years"]

X = df[FEATURES_3]
y = df["resale_price"]

# Single split reused across every experiment so comparisons are apples-to-apples
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42
)

# sparse_output=False: dense arrays are much faster for RandomForest/GB split-finding
# than scipy sparse matrices once the categorical has this many levels (town = 26).
preproc_2feat = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["town"])],
    remainder="passthrough",
)
preproc_3feat = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["town"])],
    remainder="passthrough",
)

print("=" * 70)
print("PART C BASELINE  |  floor_area_sqm + town  |  LinearRegression")
print("=" * 70)
base_pipe = Pipeline([("prep", preproc_2feat), ("model", LinearRegression())])
base_pipe.fit(X_train[FEATURES_BASE], y_train)
pred_base = base_pipe.predict(X_test[FEATURES_BASE])
mae_base = mean_absolute_error(y_test, pred_base)
r2_base = r2_score(y_test, pred_base)
print(f"TEST MAE: {mae_base:,.0f}   TEST R2: {r2_base:.4f}")

# ----------------------------------------------------------------------------
# OPTION 1 — add remaining_lease_years as a third feature
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("OPTION 1  |  + remaining_lease_years  |  LinearRegression")
print("=" * 70)
opt1_pipe = Pipeline([("prep", preproc_3feat), ("model", LinearRegression())])
opt1_pipe.fit(X_train, y_train)
pred_opt1 = opt1_pipe.predict(X_test)
mae_opt1 = mean_absolute_error(y_test, pred_opt1)
r2_opt1 = r2_score(y_test, pred_opt1)
print(f"TEST MAE: {mae_opt1:,.0f}   TEST R2: {r2_opt1:.4f}")
print(f"\n--> vs baseline: MAE {'improved' if mae_opt1 < mae_base else 'worsened'} by "
      f"{abs(mae_base - mae_opt1):,.0f}  |  R2 {'improved' if r2_opt1 > r2_base else 'worsened'} "
      f"by {abs(r2_opt1 - r2_base):.4f}")
print("Adding lease length gives the model real new information — older flats are "
      "systematically cheaper even within the same town — so both metrics improve "
      "noticeably. Because town already captures most of the location effect, this "
      "3rd feature is now competing with floor_area_sqm and lease for the remaining "
      "variance, rather than trying to substitute for missing location info.")

# ----------------------------------------------------------------------------
# OPTION 2 — RandomForestRegressor(n_estimators=300) vs 100
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("OPTION 2  |  RandomForestRegressor: n_estimators=100 vs 300")
print("=" * 70)
for n in (100, 300):
    t0 = time.time()
    rf_pipe = Pipeline(
        [("prep", preproc_3feat),
         ("model", RandomForestRegressor(
             n_estimators=n, max_depth=15, min_samples_leaf=5,
             random_state=42, n_jobs=1))]
    )
    rf_pipe.fit(X_train, y_train)
    fit_time = time.time() - t0
    pred_train = rf_pipe.predict(X_train)
    pred_test = rf_pipe.predict(X_test)
    print(f"\nn_estimators={n}  (fit time: {fit_time:.1f}s)")
    print(f"  TRAIN  MAE: {mean_absolute_error(y_train, pred_train):,.0f}   "
          f"R2: {r2_score(y_train, pred_train):.4f}")
    print(f"  TEST   MAE: {mean_absolute_error(y_test, pred_test):,.0f}   "
          f"R2: {r2_score(y_test, pred_test):.4f}")
    if n == 100:
        rf100_test_mae, rf100_test_r2, rf100_time = (
            mean_absolute_error(y_test, pred_test), r2_score(y_test, pred_test), fit_time
        )
    else:
        rf300_test_mae, rf300_test_r2, rf300_time = (
            mean_absolute_error(y_test, pred_test), r2_score(y_test, pred_test), fit_time
        )

print(f"\n--> Tripling the trees ({rf100_time:.0f}s -> {rf300_time:.0f}s, "
      f"~{rf300_time/rf100_time:.1f}x slower) moved TEST MAE by only "
      f"{abs(rf100_test_mae - rf300_test_mae):,.0f} and TEST R2 by "
      f"{abs(rf100_test_r2 - rf300_test_r2):.4f}.")
print("100 trees is already enough for the forest's predictions to stabilize -- "
      "averaging more trees mostly reduces variance, and that effect has steeply "
      "diminishing returns past the first ~100. The extra 200 trees cost real "
      "wall-clock time but bought almost nothing back in accuracy.")

# ----------------------------------------------------------------------------
# OPTION 3 — find the single most-wrong prediction
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("OPTION 3  |  Single worst-predicted flat in the test set (RF, n=100)")
print("=" * 70)
rf_pipe_100 = Pipeline(
    [("prep", preproc_3feat),
     ("model", RandomForestRegressor(
         n_estimators=100, max_depth=15, min_samples_leaf=5,
         random_state=42, n_jobs=1))]
)
rf_pipe_100.fit(X_train, y_train)
pred_test_100 = rf_pipe_100.predict(X_test)

errors = pd.DataFrame({
    "row_index": idx_test,
    "actual_price": y_test.values,
    "predicted_price": pred_test_100,
})
errors["abs_error"] = (errors["actual_price"] - errors["predicted_price"]).abs()
errors = errors.sort_values("abs_error", ascending=False)

worst = errors.iloc[0]
worst_row = df.loc[int(worst["row_index"])]
direction = "OVER-predicted" if worst["predicted_price"] > worst["actual_price"] else "UNDER-predicted"
print(f"\nActual price:    ${worst['actual_price']:,.0f}")
print(f"Predicted price: ${worst['predicted_price']:,.0f}  ({direction})")
print(f"Absolute error:  ${worst['abs_error']:,.0f}")
print("\nFull details of this flat:")
print(worst_row[["town", "flat_type", "floor_area_sqm", "storey_range",
                  "flat_model", "remaining_lease", "resale_price"]].to_string())
print("\n--> What's special: the model only knows floor_area_sqm, town, and "
      "remaining_lease. It learned that this town generally commands high prices "
      "(or low ones), but has no idea about THIS unit's storey_range or flat_model -- "
      "both of which swing price by hundreds of thousands of dollars within the same "
      "town. A flat near the ground floor of a basic model in an expensive town gets "
      "priced as if it were a premium high-floor unit there, because 'town' alone "
      "can't distinguish them. This is the same underfitting-from-missing-features "
      "story as before, just relocated to a different blind spot.")

# ----------------------------------------------------------------------------
# PART D (Stretch) — Stacking: RandomForest + GradientBoosting -> Ridge
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART D (Stretch)  |  Stacking: RF + GradientBoosting -> Ridge (manager)")
print("=" * 70)

t0 = time.time()
rf_solo = Pipeline(
    [("prep", preproc_3feat),
     ("model", RandomForestRegressor(
         n_estimators=100, max_depth=15, min_samples_leaf=5,
         random_state=42, n_jobs=1))]
)
rf_solo.fit(X_train, y_train)
pred_rf_solo = rf_solo.predict(X_test)
mae_rf_solo, r2_rf_solo = mean_absolute_error(y_test, pred_rf_solo), r2_score(y_test, pred_rf_solo)
print(f"RF alone:           MAE={mae_rf_solo:,.0f}  R2={r2_rf_solo:.4f}  ({time.time()-t0:.1f}s)")

t0 = time.time()
gb_solo = Pipeline(
    [("prep", preproc_3feat),
     ("model", GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42))]
)
gb_solo.fit(X_train, y_train)
pred_gb_solo = gb_solo.predict(X_test)
mae_gb_solo, r2_gb_solo = mean_absolute_error(y_test, pred_gb_solo), r2_score(y_test, pred_gb_solo)
print(f"GradientBoosting:   MAE={mae_gb_solo:,.0f}  R2={r2_gb_solo:.4f}  ({time.time()-t0:.1f}s)")

t0 = time.time()
base_learners = [
    ("rf", RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=5,
                                  random_state=42, n_jobs=1)),
    ("gb", GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)),
]
stack_pipe = Pipeline(
    [("prep", preproc_3feat),
     ("model", StackingRegressor(estimators=base_learners, final_estimator=Ridge(), cv=3, n_jobs=1))]
)
stack_pipe.fit(X_train, y_train)
pred_stack = stack_pipe.predict(X_test)
mae_stack, r2_stack = mean_absolute_error(y_test, pred_stack), r2_score(y_test, pred_stack)
print(f"Stacked (RF+GB):    MAE={mae_stack:,.0f}  R2={r2_stack:.4f}  ({time.time()-t0:.1f}s)")

best_solo_mae = min(mae_rf_solo, mae_gb_solo)
best_solo_r2 = max(r2_rf_solo, r2_gb_solo)
if mae_stack < best_solo_mae:
    verdict = (f"Stacking beat the best single model's MAE by {best_solo_mae - mae_stack:,.0f} "
               f"and R2 by {r2_stack - best_solo_r2:.4f}.")
else:
    verdict = (f"Stacking did NOT beat RF alone here -- it was {mae_stack - best_solo_mae:,.0f} "
               f"WORSE on MAE and {best_solo_r2 - r2_stack:.4f} worse on R2, despite taking "
               "roughly as long as fitting RF and GB separately, plus the meta-model on top.")
print(f"\n--> {verdict}")
print("With this feature set, RF (R2={:.3f}) is clearly stronger than GB (R2={:.3f}). "
      "When one base learner is much weaker than the other, blending them can drag the "
      "ensemble's predictions toward the weaker model's mistakes rather than canceling "
      "them out -- the 'manager' (Ridge) can only reweight what the two experts already "
      "got wrong; it can't invent information neither of them had. Stacking pays off most "
      "when the base learners are comparably strong AND make different kinds of errors -- "
      "that's not quite the case here, since both are tree ensembles trained on the exact "
      "same 3 features.".format(r2_rf_solo, r2_gb_solo))

print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
summary = pd.DataFrame({
    "Model": ["Baseline (2 feat) LR", "+ remaining_lease LR", "RF (100 trees)",
              "RF (300 trees)", "GradientBoosting", "Stacked (RF+GB->Ridge)"],
    "Test MAE": [mae_base, mae_opt1, rf100_test_mae, rf300_test_mae, mae_gb_solo, mae_stack],
    "Test R2": [r2_base, r2_opt1, rf100_test_r2, rf300_test_r2, r2_gb_solo, r2_stack],
})
summary["Test MAE"] = summary["Test MAE"].map(lambda v: f"{v:,.0f}")
summary["Test R2"] = summary["Test R2"].map(lambda v: f"{v:.4f}")
print(summary.to_string(index=False))

PART C BASELINE  |  floor_area_sqm + town  |  LinearRegression
TEST MAE: 108,534   TEST R2: 0.4770

OPTION 1  |  + remaining_lease_years  |  LinearRegression
TEST MAE: 96,039   TEST R2: 0.6079

--> vs baseline: MAE improved by 12,495  |  R2 improved by 0.1309
Adding lease length gives the model real new information — older flats are systematically cheaper even within the same town — so both metrics improve noticeably. Because town already captures most of the location effect, this 3rd feature is now competing with floor_area_sqm and lease for the remaining variance, rather than trying to substitute for missing location info.

OPTION 2  |  RandomForestRegressor: n_estimators=100 vs 300

n_estimators=100  (fit time: 26.9s)
  TRAIN  MAE: 62,814   R2: 0.8038
  TEST   MAE: 64,904   R2: 0.7919

n_estimators=300  (fit time: 82.0s)
  TRAIN  MAE: 62,746   R2: 0.8040
  TEST   MAE: 64,864   R2: 0.7921

--> Tripling the trees (27s -> 82s, ~3.0x slower) moved TEST MAE by only 39 and TEST R2 by 0.00